In [10]:
# %%
# نوتبوک: 06_test_main_graph_with_tools.ipynb
# هدف: تست end-to-end گراف مولتی‌ایجنت (researcher + reasoner + critic)

import os
import sys
from pathlib import Path

# تنظیم مسیر پروژه
project_root = Path.cwd().parent  # اگر نوتبوک در notebooks است
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"✓ Project root: {project_root}")
print(f"✓ Src path: {src_path}")

from dotenv import load_dotenv
load_dotenv()

print("✓ OPENROUTER_API_KEY:", (os.getenv("OPENROUTER_API_KEY") or "")[:20], "...")
print("✓ COHERE_API_KEY:", (os.getenv("COHERE_API_KEY") or "")[:20], "...")


✓ Project root: f:\Thesis\project\3-Multi-Agent-System
✓ Src path: f:\Thesis\project\3-Multi-Agent-System\src
✓ OPENROUTER_API_KEY: sk-or-v1-10d2ef641ff ...
✓ COHERE_API_KEY: rq8LfpFUvkclAvSyxHfD ...


In [11]:
# %%
from legal_multi_agent.graphs.main_graph import graph
from legal_multi_agent.state.constants import CATEGORY_TO_DOMAIN

print("✓ Graph imported")
print("✓ Categories:", list(CATEGORY_TO_DOMAIN.keys()))


✓ Graph imported
✓ Categories: ['حقوق مدنی', 'آیین دادرسی مدنی', 'حقوق تجارت', 'حقوق جزا', 'آیین دادرسی کیفری', 'حقوق اساسی']


In [ ]:
# %%
from pprint import pprint

def run_mcq_through_graph(
    question_number: int,
    category: str,
    question: str,
    options_text: str,
    max_revisions: int = 2,
):
    domain = CATEGORY_TO_DOMAIN.get(category, "civil")
    
    initial_state = {
        "question_number": question_number,
        "category": category,
        "domain": domain,
        "question": question,
        "options_text": options_text,
        "max_revisions": max_revisions,
        "use_option_verifier": False,
        "use_retriever_tool": False,
    }
    
    print("="*70)
    print(f"سوال {question_number} - دسته: {category} | دامین: {domain}")
    print("="*70)
    print("سوال:")
    print(question)
    print("\nگزینه‌ها:")
    print(options_text)
    print("\n⏳ در حال اجرای گراف...")
    
    result = graph.invoke(initial_state)
    
    print("\n✅ اجرای گراف تمام شد.")
    
    final_toon = result.get("final_toon") or {}
    draft_toon = result.get("draft_toon") or {}
    critic_toon = result.get("critic_toon") or {}
    
    print("\n--- نتیجه نهایی (final_toon) ---")
    pprint(final_toon)
    
    print("\n--- آخرین پیش‌نویس (draft_toon) ---")
    pprint(draft_toon)
    
    print("\n--- نظر منتقد (critic_toon) ---")
    pprint(critic_toon)
    
    print("\n--- اطلاعات کمکی ---")
    print("revision_count:", result.get("revision_count"))
    print("max_revisions:", result.get("max_revisions"))
    
    return result


In [13]:
# %%
q1 = "شرط وکالت زن در طلاق در چه قالب هایی قابل تحقق است؟"

opts1 = """
"1) به شکل شرط نتیجه یا شرط فعل، ضمن عقد نکاح"\n
      "2) به شکل شرط فعل، ضمن عقد نکاح یا عقد دیگری"\n
      "3) به شکل شرط نتیجه یا فعل ضمن عقد نکاح یا عقد دیگری"\n
      "4) به شکل شرط نتیجه ضمن عقد نکاح یا عقد دیگری"
"""

res1 = run_mcq_through_graph(
    question_number=5,
    category="حقوق مدنی",
    question=q1,
    options_text=opts1,
)


سوال 5 - دسته: حقوق مدنی | دامین: civil
سوال:
شرط وکالت زن در طلاق در چه قالب هایی قابل تحقق است؟

گزینه‌ها:

"1) به شکل شرط نتیجه یا شرط فعل، ضمن عقد نکاح"

      "2) به شکل شرط فعل، ضمن عقد نکاح یا عقد دیگری"

      "3) به شکل شرط نتیجه یا فعل ضمن عقد نکاح یا عقد دیگری"

      "4) به شکل شرط نتیجه ضمن عقد نکاح یا عقد دیگری"


⏳ در حال اجرای گراف...
📝 Query: شرط وکالت زن در طلاق در چه قالب هایی قابل تحقق است؟...
🔍 روش انتخاب شده: Semantic search
🔍 Semantic only: 20 نتیجه
   ✓ Retrieved: 20 documents
🔄 Reranking 20 سند (بدون law+article)...
   ✓ Reranked: top 5 documents

✅ اجرای گراف تمام شد.

--- نتیجه نهایی (final_toon) ---
{'answer': '3',
 'confidence': 5,
 'explanation': 'بر اساس ماده 1119 قانون مدنی، شرط وکالت زن در طلاق '
                'می\u200cتواند به صورت شرط نتیجه یا شرط فعل در ضمن عقد نکاح یا '
                'عقد لازم دیگری قرار گیرد، مشروط بر اینکه مخالف مقتضای عقد '
                'نباشد.'}

--- آخرین پیش‌نویس (draft_toon) ---
{'answer': '3',
 'confidence': 5,
 'expl

In [20]:
# %%
q2 = """
کدام مجازات تعزیری زیر در مقام تخفیف قابل تبدیل به مجازات دیگر نیست؟
"""

opts2 = """
1) مصادره اموال
2) محرومیت از حقوق اجتماعی 
3) حبس درجه 3 
4) شلاق درجه 6
"""

res2 = run_mcq_through_graph(
    question_number=2,
    category="حقوق جزا",
    question=q2,
    options_text=opts2,
)


سوال 2 - دسته: حقوق جزا | دامین: criminal
سوال:

کدام مجازات تعزیری زیر در مقام تخفیف قابل تبدیل به مجازات دیگر نیست؟


گزینه‌ها:

1) مصادره اموال
2) محرومیت از حقوق اجتماعی 
3) حبس درجه 3 
4) شلاق درجه 6


⏳ در حال اجرای گراف...
📝 Query: 
کدام مجازات تعزیری زیر در مقام تخفیف قابل تبدیل به مجازات دیگر نیست؟
...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون کاهش مجازات حبس تعزیری' | None None
🔍 فیلتر فقط قانون...
   ✓ 10 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents

✅ اجرای گراف تمام شد.

--- نتیجه نهایی (final_toon) ---
{'answer': '4',
 'confidence': 5,
 'explanation': 'مطابق ماده 37 قانون مجازات اسلامی، مجازات\u200cهای تعزیری '
                'قابل تخفیف یا تبدیل شامل حبس، جزای نقدی، مصادره و انفصال '
                'هستند، اما شلاق درجه 6 جزو مجازات\u200cهای تعزیری قانونی است '
                'و قابل تبدیل به مجازات دیگر نیست؛ زیرا شلاق در جرایم تعزیری '
      

In [19]:
# %%
q3 = """
مجازات معاون در اسیدپاشی چه مقدار است؟ 
"""

opts2 = """
1) به حبس درجه 1 یا 2 محکوم می شود. 
2) مشابه مرتکب جرم مجازات میشود. 
3) به مجازات های خاص مقرر در قانون مجازات اسیدپاشی و حمایت از بزه دیدگان ناشی از آن محکوم می شود . 
4) طبق قواعد عمومی معاونت در قانون مجازات اسلامی مجازات میشود. 
"""

res3 = run_mcq_through_graph(
    question_number=3,
    category="حقوق جزا",
    question=q3,
    options_text=opts2,
)


سوال 3 - دسته: حقوق جزا | دامین: criminal
سوال:

مجازات معاون در اسیدپاشی چه مقدار است؟ 


گزینه‌ها:

1) به حبس درجه 1 یا 2 محکوم می شود. 
2) مشابه مرتکب جرم مجازات میشود. 
3) به مجازات های خاص مقرر در قانون مجازات اسیدپاشی و حمایت از بزه دیدگان ناشی از آن محکوم می شود . 
4) طبق قواعد عمومی معاونت در قانون مجازات اسلامی مجازات میشود. 


⏳ در حال اجرای گراف...
📝 Query: 
مجازات معاون در اسیدپاشی چه مقدار است؟ 
...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون تشدید مجازات اسیدپاشی و حمایت از بزه‌دیدگان ناشی از آن' | None None
🔍 فیلتر فقط قانون...
   ✓ 7 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents

✅ اجرای گراف تمام شد.

--- نتیجه نهایی (final_toon) ---
{'answer': '2',
 'confidence': 5,
 'explanation': 'مطابق ماده 1 قانون تشدید مجازات اسیدپاشی و حمایت از '
                'بزه\u200cدیدگان ناشی از آن، در موارد اسیدپاشی، مرتکب به قصاص '
                'محکوم می\u200cشو

In [21]:
# %%
def print_rag_debug(result, limit: int = 3):
    print("="*70)
    print("RAG Debug - اولین اسناد")
    print("="*70)
    meta_list = result.get("docs_meta", []) or result.get("rag_results", [])
    for meta in meta_list[:limit]:
        print(meta)
    print("\nPreview context:")
    print((result.get("context_preview") or result.get("context") or "")[:800])

print_rag_debug(res1)
print_rag_debug(res2)
print_rag_debug(res3)


RAG Debug - اولین اسناد
{'i': 1, 'law': 'قانون مدنی', 'article_number': None, 'source_type': 'وحدت رویه', 'title': 'عدم امکان اعمال وکالت زوجه در طلاق در صورت اثبات عدم تمکین بدون مانع مشروع'}
{'i': 2, 'law': 'قانون حمایت خانواده | قانون مدنی', 'article_number': None, 'source_type': 'دادنامه', 'title': 'اثر پرداخت نفقه معوقه زوجه متعاقب طرح دعوی طلاق از ناحیه وی'}
{'i': 3, 'law': 'قانون مدنی', 'article_number': 1119, 'source_type': 'قانون', 'title': None}

Preview context:
[منبع 1] (وحدت رویه) - قانون مدنی
title: عدم امکان اعمال وکالت زوجه در طلاق در صورت اثبات عدم تمکین بدون مانع مشروع
issuer: هیات‌ عمومی دیوان ‌عالی ‌کشور
vote_text: نظر به اینکه مطابق ماده 1108 قانون مدنی تمکین از زوج تکلیف قانونی زوجه است، بنابراین در صورتی که بدون مانع مشروع از ادای وظایف زوجیت امتناع و زوج این امر را در دادگاه اثبات و با اخذ اجازه از دادگاه همسر دیگری اختیار نماید، وکالت زوجه از زوج در طلاق که به حکم ماده 1119 قانون مدنی ضمن عقد نکاح شرط و مراتب در سند ازدواج ذیل بند ب شرایط ضمن عقد در ردیف 12 قید